# 01. Fakeddit Preprocessing

This notebook reproduces the full preprocessing pipeline used to construct the **HARFM** dataset from the raw Fakeddit corpus.

## Pipeline Overview
1. **Merge** three Fakeddit splits (train / validate / test_public)
2. **Image filtering** — remove rows without a valid image URL or `hasImage=False`
3. **Length filtering** — IQR-based outlier removal + below-mean removal + `propagandaposters` exclusion
4. **Sampling** — allocate HR / HF / AR-source / AF-source pools

## Expected Input Files
Place the following files in the **same directory** as this notebook:
- `multimodal_train.tsv`
- `multimodal_validate.tsv`
- `multimodal_test_public.tsv`

> Download from the official [Fakeddit repository](https://github.com/entitize/Fakeddit).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(".")
TSV_FILES = [
    DATA_DIR / "multimodal_train.tsv",
    DATA_DIR / "multimodal_validate.tsv",
    DATA_DIR / "multimodal_test_public.tsv",
]
SEED = 42
TOTAL_AI_TARGET = 100_000
RATIO_REAL = 0.6
RATIO_FAKE = 0.4

## Stage 0 — Merge Raw Fakeddit Splits

In [ ]:
dfs = []
for f in TSV_FILES:
    df = pd.read_csv(f, sep="\t", low_memory=False)
    df["_source"] = f.stem
    dfs.append(df)
    print(f"  {f.name}: {len(df):,} rows")

df_all = pd.concat(dfs, ignore_index=True)
df_all["2_way_label"] = df_all["2_way_label"].map({0: "fake", 1: "real"})

r0 = (df_all["2_way_label"] == "real").sum()
f0 = (df_all["2_way_label"] == "fake").sum()
print(f"\n[Stage 0] Merged: total={len(df_all):,}  real={r0:,}  fake={f0:,}")

## Stage 1 — Image Filtering

Remove samples that have no valid `image_url` or `hasImage=False`.
These correspond to deleted Reddit posts or broken image links.

In [ ]:
has_valid_url  = df_all["image_url"].notna() & (df_all["image_url"].astype(str).str.strip() != "")
has_image_flag = (df_all["hasImage"] == True) | (df_all["hasImage"].astype(str).str.lower() == "true")
df_img = df_all[has_valid_url & has_image_flag].reset_index(drop=True)

r1 = (df_img["2_way_label"] == "real").sum()
f1 = (df_img["2_way_label"] == "fake").sum()
print(f"[Stage 1] After image filter: total={len(df_img):,}  real={r1:,}  fake={f1:,}")
print(f"          Removed: real={r0-r1:,}  fake={f0-f1:,}  total={len(df_all)-len(df_img):,}")

## Stage 2 — Headline Length Filtering

Three sub-steps applied sequentially:
1. **IQR-based outlier removal** — drop headlines outside `[Q1 − 1.5×IQR, Q3 + 1.5×IQR]`
2. **Below-mean removal** — drop headlines shorter than the post-IQR mean length
3. **`propagandaposters` exclusion** — this subreddit's image–text veracity labels are systematically inconsistent with the rest of the corpus (all samples are labelled *fake* but the images are genuine propaganda posters)

In [ ]:
df_img["title_len"] = df_img["clean_title"].astype(str).str.len()

q1, q3 = df_img["title_len"].quantile([0.25, 0.75])
iqr     = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR bounds: Q1={q1:.1f}, Q3={q3:.1f}, IQR={iqr:.1f}  →  [{low:.1f}, {high:.1f}]")

df_iqr   = df_img[(df_img["title_len"] >= low) & (df_img["title_len"] <= high)].copy()
mean_len = df_iqr["title_len"].mean()
print(f"Mean length after IQR filter: {mean_len:.1f} chars")

df_filtered = df_iqr[
    (df_iqr["title_len"] >= mean_len) &
    (df_iqr["subreddit"] != "propagandaposters")
].reset_index(drop=True)

r2 = (df_filtered["2_way_label"] == "real").sum()
f2 = (df_filtered["2_way_label"] == "fake").sum()
print(f"\n[Stage 2] After length filter: total={len(df_filtered):,}  real={r2:,}  fake={f2:,}")
print(f"          Removed: real={r1-r2:,}  fake={f1-f2:,}  total={len(df_img)-len(df_filtered):,}")

prop = df_iqr[df_iqr["subreddit"] == "propagandaposters"]
print(f"\n  [Note] propagandaposters removed: {len(prop):,} rows  "
      f"(real={( prop['2_way_label']=='real').sum():,}, fake={(prop['2_way_label']=='fake').sum():,})")

## Stage 3 — Sampling (HR / HF / AR-source / AF-source)

From the filtered pool we allocate:
- **AR-source**: 60,000 samples drawn from the *real* pool (used as seeds for AI rewriting → AR label)
- **AF-source**: 40,000 samples (20,000 from *real* + 20,000 from *fake*) used as seeds for AI fake generation → AF label
- **HR**: remaining *real* samples kept as human-authored real headlines
- **HF**: subsampled *fake* pool to maintain an overall 6:4 real-to-fake ratio

In [ ]:
hr_pool = df_filtered[df_filtered["2_way_label"] == "real"].copy()
hf_pool = df_filtered[df_filtered["2_way_label"] == "fake"].copy()
print(f"Available pool: HR(real)={len(hr_pool):,}  HF(fake)={len(hf_pool):,}")

n_ar = int(TOTAL_AI_TARGET * RATIO_REAL)   # 60,000
n_af = int(TOTAL_AI_TARGET * RATIO_FAKE)   # 40,000

df_ar_src = hr_pool.sample(n=n_ar, random_state=SEED)
hr_pool   = hr_pool.drop(df_ar_src.index)

n_af_hr, n_af_hf = n_af // 2, n_af - (n_af // 2)
af_hr_src = hr_pool.sample(n=n_af_hr, random_state=SEED+1)
hr_pool   = hr_pool.drop(af_hr_src.index)
af_hf_src = hf_pool.sample(n=n_af_hf, random_state=SEED+2)
hf_pool   = hf_pool.drop(af_hf_src.index)

df_hr_final = hr_pool.copy()
total_real  = len(df_hr_final) + n_ar
n_hf_needed = int(total_real / 1.5 - (n_af_hr + n_af_hf))
df_hf_final = hf_pool.sample(n=n_hf_needed, random_state=SEED+3)

print(f"\n[Stage 3] Final HARFM label counts:")
print(f"  HR = {len(df_hr_final):,}")
print(f"  HF = {len(df_hf_final):,}")
print(f"  AR-source = {n_ar:,}")
print(f"  AF-source = {n_af_hr + n_af_hf:,}  (from real: {n_af_hr:,}, from fake: {n_af_hf:,})")
print(f"  Total = {len(df_hr_final)+len(df_hf_final)+n_ar+n_af_hr+n_af_hf:,}")

## Preprocessing Summary Table

| Stage | Real | Fake | Total |
|---|---|---|---|
| Original Fakeddit (3 splits) | 268,908 | 413,753 | 682,661 |
| After image filtering | 267,601 | 413,197 | 680,798 |
| → Removed | 1,307 | 556 | 1,863 |
| After length filtering | 164,954 | 110,156 | 275,110 |
| → Removed | 102,647 | 303,041 | 405,688 |
| Final HR / HF (human) | 84,954 | 56,636 | 141,590 |
| AR / AF source (AI-generated) | 60,000 | 40,000 | 100,000 |
| **HARFM total** | **144,954** | **96,636** | **241,590** |

## Save Base CSV for Generation Step

In [ ]:
df_ar_src["4_way_label"] = "AR";  df_ar_src["ai_source"] = "HR"
af_hr_src["4_way_label"] = "AF";  af_hr_src["ai_source"] = "HR"
af_hf_src["4_way_label"] = "AF";  af_hf_src["ai_source"] = "HF"
df_hr_final["4_way_label"] = "HR"; df_hr_final["ai_source"] = ""
df_hf_final["4_way_label"] = "HF"; df_hf_final["ai_source"] = ""

df_af_pool = pd.concat([af_hr_src, af_hf_src])
df_base = pd.concat([df_hr_final, df_hf_final, df_ar_src, df_af_pool])
df_base = df_base.sample(frac=1, random_state=SEED).reset_index(drop=True)

out_path = DATA_DIR / "HARFM_Step_1.csv"
df_base.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(df_base):,} rows)")
print(df_base["4_way_label"].value_counts())